# Periodic Ising evolution terms

The one-dimensional model has a transverse field and a periodic nearest-neighbor
interaction, $H=-J\sum_j Z_j Z_{(j+1)\bmod L}-\Gamma\sum_j X_j$.
This notebook follows the two three-site examples and their immediate calls.
Each term binds to an ordered system register.

Adapted from the [Qualtran v0.7.0 notebook](https://github.com/quantumlib/Qualtran/blob/bcb17bb25360ed68ce3bc084645db3e929799389/qualtran/bloqs/chemistry/trotter/ising/ising.ipynb), Copyright 2023 Google LLC, Apache-2.0.

In [ ]:
from IPython.display import Markdown, SVG, display

from flagquantum.algorithms.primitives.ising_evolution import IsingXUnitary, IsingZZUnitary
from flagquantum.circuit import Circuit
from flagquantum.compiler._component_trace.call_graph import modeled_call_graph
from flagquantum.drawer.call_graph import (
    export_modeled_call_graph_dot, format_call_graph_markdown, render_call_graph_svg,
)
from flagquantum.drawer.signature_graph import export_signature_graph_dot
from flagquantum.qec.resources.estimation import estimate_ftqc_resources


def show_example(term, angle, label):
    model = term.resource_model(angle)
    graph = modeled_call_graph(model, max_depth=1)
    for dot in (
        export_signature_graph_dot(term.signature, label, pretty=True),
        export_modeled_call_graph_dot(model, max_depth=1),
    ):
        display(SVG(data=render_call_graph_svg(dot).decode()))
    display(Markdown(format_call_graph_markdown(
        graph.root, graph.direct_calls, graph.leaf_counts, title=lambda node: node.name
    )))
    return {node.name: count for node, count in graph.leaf_counts.items()}

## Transverse field

The example binds three sites and rotation angle 0.04. Each site receives
one Rx in register order.

In [ ]:
sites, coupling, timestep = 3, 2, 0.01
angle = 2 * coupling * timestep
x = IsingXUnitary(sites)
assert show_example(x, angle, "U_X") == {"Rx": 3}
x_op = x.make_on(system=(0, 1, 2), angle=angle)
x_circuit = Circuit(sites).append(x_op)
assert [(g.name, g.wires, g.params["theta"]) for g in x_circuit.to_ir().instructions] == [
    ("rx", (wire,), angle) for wire in range(3)
]

## Periodic ZZ term

Bind the same three sites and angle. A pair of controlled-NOTs surrounds
an Rz on the target of each directed edge, including the edge from the
last site back to the first.

In [ ]:
zz = IsingZZUnitary(sites)
assert show_example(zz, angle, "U_ZZ") == {"CNOT": 6, "Rz": 3}
zz_op = zz.make_on(system=(0, 1, 2), angle=angle)
zz_circuit = Circuit(sites).append(zz_op)
gates = zz_circuit.to_ir().instructions
assert [g.wires for g in gates[::3]] == [(0, 1), (1, 2), (2, 0)]
assert all(g.params["theta"] == angle for g in gates[1::3])
report = estimate_ftqc_resources(zz_op, emitted_ir=zz_circuit.to_ir())
assert report.emitted is not None
assert report.emitted.unconditional == {"cx": 6, "rz": 3}
assert report.logical.gates.rotation == 3
assert report.logical.peak_logical_qubits == 3
print("Emitted gates:", report.emitted.unconditional)
print("Modeled peak logical qubits:", report.logical.peak_logical_qubits)

## Two-site boundary

The periodic bonds are directed by site order. At size two both directions
remain in the original schedule, so the physical sequence contains two
phases for the same unordered site pair. Periodic ZZ requires at least
two distinct sites. These circuits use no work or measurement wires.

In [ ]:
two_site = Circuit(2).append(IsingZZUnitary(2).make_on(system=(1, 0), angle=angle))
assert [g.wires for g in two_site.to_ir().instructions[::3]] == [(1, 0), (0, 1)]